# Notebook 2: Chunking Strategy Variation
## Description:
This notebook is an A/B test focusing on the chunking strategy. It evaluates the performance of a basic fixed-length chunking approach compared to the baseline's contextual chunking. All other components remain identical to the baseline.
## Pipeline Configuration:
Input Data: Cleaned text dataset  
**Chunking: Fixed-length chunking (500 characters) ️ [Variable Changed]**  
Embedding: BAAI method  
Retrieval: L2 Distance  
Prompt & Generation: Context method  

In [3]:
# Executes a baseline fixed-size chunking strategy on the master corpus.
import json

def baseline_fixed_chunking(input_file, output_file):
    print("Initializing baseline fixed-size chunking process...")
    
    CHUNK_SIZE = 800
    OVERLAP = 100
    
    with open(input_file, 'r', encoding='utf-8') as f:
        corpus = json.load(f)

    all_chunks = []
    chunk_counter = 1

    for doc in corpus:
        title = doc.get("title", "Unknown Title")
        source = doc.get("source", "Unknown Source")
        sections = doc.get("content_sections", {})
        
        for heading, content in sections.items():
            text = str(content).strip()
            
            if not text:
                continue
            
            start = 0
            text_length = len(text)
            
            while start < text_length:
                end = min(start + CHUNK_SIZE, text_length)
                chunk_text = text[start:end]
                
                chunk_doc = {
                    "chunk_id": f"baseline_chunk_{chunk_counter:04d}",
                    "text": chunk_text,
                    "metadata": {
                        "source": source,
                        "title": title
                    }
                }
                all_chunks.append(chunk_doc)
                chunk_counter += 1
                
                start += (CHUNK_SIZE - OVERLAP)

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=4)
        
    print(f"      [Success] Chunking complete. Generated {len(all_chunks)} fixed-size chunks.")
    print(f"Data saved to: {output_file}\n")

if __name__ == "__main__":
    input_file = "Background_Corpus.json"
    output_file = "Baseline_rag_chunks.json"
    
    baseline_fixed_chunking(input_file, output_file)

Initializing baseline fixed-size chunking process...
      [Success] Chunking complete. Generated 1426 fixed-size chunks.
Data saved to: Baseline_rag_chunks.json



In [4]:
# Vectorises baseline chunks using BAAI/bge-small-en-v1.5 and builds a FAISS vector index.
import json
import faiss
from sentence_transformers import SentenceTransformer

with open('Baseline_rag_chunks.json', 'r', encoding='utf-8') as f:
    chunks = json.load(f)

texts = [chunk["text"] for chunk in chunks]
print(f"Successfully loaded {len(texts)} baseline chunks.")

MODEL_NAME = 'BAAI/bge-small-en-v1.5'
print(f"Loading embedding model: {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)

print("Vectorising text data...")
embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

print(f"Vectorisation complete. Embedding matrix shape: {embeddings.shape}")

print("Building FAISS index...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

index.add(embeddings)
print(f"Successfully added {index.ntotal} vectors to the FAISS index.")

faiss.write_index(index, 'Baseline_rag_vector.index')
print("Vector store saved to: Baseline_rag_vector.index")

Successfully loaded 1426 baseline chunks.
Loading embedding model: BAAI/bge-small-en-v1.5...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorising text data...


Batches:   0%|          | 0/45 [00:00<?, ?it/s]

Vectorisation complete. Embedding matrix shape: (1426, 384)
Building FAISS index...
Successfully added 1426 vectors to the FAISS index.
Vector store saved to: Baseline_rag_vector.index


In [7]:
# Executes baseline FAISS dense retrieval for input queries and exports top-K contexts to an intermediate JSON file.
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

def run_retrieval_step(qa_file_path, index_path, chunks_file_path, intermediate_output_path, k=5):
    print("Initiating RAG retrieval process...")
    
    print("Loading dataset and vectorising queries...")
    with open(qa_file_path, 'r', encoding='utf-8') as f:
        qa_data = json.load(f)
        
    queries = []
    gold_answers = []
    for source_group in qa_data.get("sources", []):
        for qa_pair in source_group.get("questions", []):
            queries.append(qa_pair["question"])
            gold_answers.append(qa_pair["answer"])
            
    embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
    instruction = "Represent this sentence for searching relevant passages: "
    queries_with_instruction = [instruction + q for q in queries]
    
    query_vectors = embedder.encode(queries_with_instruction, show_progress_bar=False)
    query_vectors = np.array(query_vectors).astype('float32')
    faiss.normalize_L2(query_vectors)
    print("Query vectorisation completed.")

    print("Executing FAISS dense retrieval...")
    index = faiss.read_index(index_path)
    with open(chunks_file_path, 'r', encoding='utf-8') as f:
        chunks_data = json.load(f)
        
    distances, indices = index.search(query_vectors, k)
    
    pipeline_results = []
    for i, query in enumerate(queries):
        retrieved_contexts = []
        for rank, idx in enumerate(indices[i]):
            if idx != -1 and idx < len(chunks_data):
                chunk_info = chunks_data[idx]
                retrieved_contexts.append({
                    "rank": rank + 1,
                    "text": chunk_info["text"],
                    "source": chunk_info.get("metadata", {}).get("source", "Unknown"),
                    "similarity_score": float(np.inner(query_vectors[i], index.reconstruct(int(idx)))) 
                })
                
        pipeline_results.append({
            "query_id": f"q_{i+1:03d}",
            "query": query,
            "gold_answer": gold_answers[i],
            "retrieved_contexts": retrieved_contexts
        })
        
    with open(intermediate_output_path, 'w', encoding='utf-8') as f:
        json.dump(pipeline_results, f, ensure_ascii=False, indent=4)
            
    print(f"Retrieval successfully completed. Output saved to: {intermediate_output_path}")

if __name__ == "__main__":
    INPUT_QA_FILE = "RAG_Benchmark_Dataset.json"         
    INDEX_FILE = "Baseline_rag_vector.index"
    CHUNKS_FILE = "Baseline_rag_chunks.json"
    INTERMEDIATE_FILE = "Intermediate_retrieval_2.json"               
    
    run_retrieval_step(INPUT_QA_FILE, INDEX_FILE, CHUNKS_FILE, INTERMEDIATE_FILE, k=5)

Initiating RAG retrieval process...
Loading dataset and vectorising queries...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query vectorisation completed.
Executing FAISS dense retrieval...
Retrieval successfully completed. Output saved to: Intermediate_retrieval_2.json


In [8]:
# Constructs QA prompts from retrieved contexts and saves them to a ready-to-use JSON file.
import json

def construct_prompts(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as f:
        pipeline_results = json.load(f)

    for item in pipeline_results:
        contexts = [chunk["text"] for chunk in item.get("retrieved_contexts", [])]
        context_text = "\n\n---\n\n".join(contexts)

        system_prompt = "You are a reading comprehension AI. Read the text and answer the question briefly."
        user_prompt = f"<context>\n{context_text}\n</context>\n\nBased ONLY on the context above, answer the following question:\nQuestion: {item['query']}\nAnswer:"

        item["messages"] = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(pipeline_results, f, ensure_ascii=False, indent=4)

if __name__ == "__main__":
    INTERMEDIATE_FILE = "Intermediate_retrieval_2.json"
    PROMPTS_FILE = "Ready_prompts_2.json"
    
    construct_prompts(INTERMEDIATE_FILE, PROMPTS_FILE)

In [1]:

import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

def generate_answers(input_path, output_path):
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Current computational device in use: {device}")
    if device == "cpu":
        print("WARNING: No available GPU detected.")
        
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        attn_implementation="eager"
    ).to(device)

    with open(input_path, 'r', encoding='utf-8') as f:
        pipeline_results = json.load(f)
        
    print("Starting answer generation...")
    for item in tqdm(pipeline_results, desc="Processing Progress", unit="item"):
        messages = item.get("messages", [])
        if not messages:
            continue

        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        item["generated_answer"] = response

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(pipeline_results, f, ensure_ascii=False, indent=4)

if __name__ == "__main__":
    PROMPTS_FILE = "Ready_prompts_2.json"
    FINAL_OUTPUT_FILE = "Rag_result_2.json"
    
    generate_answers(PROMPTS_FILE, FINAL_OUTPUT_FILE)

c:\Program Files\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


Current computational device in use: cuda


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 366.61it/s]
c:\Program Files\Python311\Lib\site-packages\torch\cuda\__init__.py:287: UserWarning: 
NVIDIA GeForce RTX 5060 Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90 compute_37.
If you want to use the NVIDIA GeForce RTX 5060 Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


Starting answer generation...


Processing Progress: 100%|██████████| 300/300 [07:36<00:00,  1.52s/item] 
